# European Film Data Analysis
## From Analytics to Action — DTU Spring 2026

**Case company:** Publikum (formerly Will & Agency)  
**Dataset:** 2,000 European movies from IMDb  
**Purpose:** Explore the landscape of European cinema through data to support strategic decision-making for film positioning, audience targeting, and market strategy.

---

### How to read this notebook

This notebook is organized as an **Exploratory Data Analysis (EDA)** journey. Following Tukey (1977), we let the data guide our hypotheses rather than testing preconceived ideas. Each section includes:

- **Context** — why we are doing this analysis
- **Code** — the technical implementation (with comments for beginners)
- **Interpretation** — what we learn and what limitations to keep in mind

> *"The greatest value of a picture is when it forces us to notice what we never expected to see."* — John Tukey, 1977

### Course frameworks we apply

| Framework | Source | How we use it |
|-----------|--------|---------------|
| Datafication (4 moments) | Flyverbom & Madsen (2015) | Reflect on how this data was produced, structured, distributed |
| Exploratory Data Analysis | Sapienza & Lehmann (2021) | Let visualizations guide hypothesis formation |
| Visual Network Analysis | Venturini et al. (2021) | Explore relational structures in the data |
| Critical data awareness | Mejias & Couldry (2019) | Acknowledge biases, limitations, whose interests data serves |

### Publikum's three pillars (for context)

Our dataset covers only the **References** pillar of Publikum's methodology:
1. **Story** — audience reactions to narratives *(not in our data)*
2. **Zeitgeist** — cultural relevance via social media monitoring *(not in our data)*
3. **References** — film and TV metadata for comparable-title analysis *(this is what we have)*

## Table of Contents

1. [Setup and Imports](#1-setup)
2. [Loading the Data](#2-loading)
3. [First Look: Understanding the Dataset](#3-first-look)
4. [Data Cleaning and Preparation](#4-cleaning)
5. [Genre Analysis](#5-genres)
6. [Country and Market Analysis](#6-countries)
7. [Ratings and Popularity](#7-ratings)
8. [Temporal Trends](#8-temporal)
9. [Runtime Analysis](#9-runtime)
10. [Keyword and Theme Analysis](#10-keywords)
11. [Co-production Network Analysis](#11-network)
12. [Genre Co-occurrence Network](#12-genre-network)
13. [Comparable Title Analysis](#13-comparable)
14. [Summary of Findings and Limitations](#14-summary)

<a id="1-setup"></a>
## 1. Setup and Imports

Before we begin, we load the Python libraries we need. If you are running this for the first time, install the required packages:

```bash
pip install pandas matplotlib seaborn networkx openpyxl
```

We also import helper functions from the project's `src/` folder:
- `set_style()` — applies a clean, consistent visual style to all our plots
- `save_figure()` — saves figures to the `reports/figures/` directory
- `find_comparable_titles()` — finds top-rated titles matching given genres

In [ ]:
# Standard libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import networkx as nx
from collections import Counter
from itertools import combinations
import os
import warnings
warnings.filterwarnings('ignore')

# Change to project root so all relative paths work consistently
# (data in 03-data/, figures saved to reports/figures/, utilities in src/)
os.chdir(os.path.join(os.path.dirname(os.path.abspath('.')), ''))
if os.path.basename(os.getcwd()) == 'notebooks':
    os.chdir('..')

# Create the figures output directory if it doesn't exist
os.makedirs('reports/figures', exist_ok=True)

# Project utilities — these live in the src/ folder of our project
from src.visualize import set_style, save_figure
from src.analysis import find_comparable_titles

# Apply consistent plot style across all visualizations
set_style()

print(f"Working directory: {os.getcwd()}")
print("All libraries loaded successfully!")

<a id="2-loading"></a>
## 2. Loading the Data

Our dataset is an Excel file containing 2,000 European movies sourced from IMDb. Let us load it and take a first look.

> **Datafication moment — Production:** This data originates from IMDb, a user-contributed, Amazon-owned database. Every entry reflects categorization decisions: what counts as a "genre," what is a title's "main country," which actors are "top five." As Bowker (2008) reminds us, *"raw data is an oxymoron."*

In [ ]:
# Load the dataset from the Excel file
df = pd.read_excel('03-data/European_data_2000.xlsx')

# How many rows and columns do we have?
print(f"Dataset shape: {df.shape[0]:,} movies x {df.shape[1]} columns")

<a id="3-first-look"></a>
## 3. First Look: Understanding the Dataset

Before doing any analysis, we need to understand what we are working with:
- What columns exist and what they contain
- What data types are present
- How much data is missing
- A few example rows

This step is fundamental to EDA — we need to **know our data** before we can ask questions of it.

In [ ]:
# Display all column names, their data types, and a sample value
print("=== Column Overview ===\n")
for col in df.columns:
    sample = df[col].dropna().iloc[0] if df[col].notna().any() else "N/A"
    # Truncate long sample values for readability
    sample_str = str(sample)[:60] + "..." if len(str(sample)) > 60 else str(sample)
    print(f"  {col:20s}  {str(df[col].dtype):10s}  Example: {sample_str}")

In [ ]:
# Show the first 5 rows to get a feel for the data
df.head()

In [ ]:
# Summary statistics for numeric columns
# This gives us the range, mean, and spread of numeric values
df.describe()

In [ ]:
# Check for missing values across all columns
missing = df.isnull().sum()
missing_pct = (missing / len(df) * 100).round(1)
missing_df = pd.DataFrame({'Missing Count': missing, 'Missing %': missing_pct})
missing_df = missing_df[missing_df['Missing Count'] > 0].sort_values('Missing %', ascending=False)

print("=== Columns with Missing Values ===\n")
if len(missing_df) > 0:
    print(missing_df.to_string())
else:
    print("No missing values found!")

print(f"\n\nTotal rows: {len(df):,}")
print(f"Complete rows (no missing values): {df.dropna().shape[0]:,}")

<a id="4-cleaning"></a>
## 4. Data Cleaning and Preparation

Several columns contain **comma-separated strings** (genres, countries, keywords, actors, directors, writers, production companies). To analyze these properly, we need to parse them into individual items. We also create some **derived columns** that will be useful later.

> **Datafication moment — Structuring:** Every cleaning decision we make is an analytical choice. When we split "Drama, Thriller" into two separate genres, we treat them as independent — but a "Drama-Thriller" might be a distinct category in the minds of filmmakers and audiences. Be aware that our structuring shapes what we can see.

In [ ]:
def split_field(series, sep=', '):
    """
    Split a comma-separated string column into a flat list of all individual values.
    
    For example, if a column contains:
        "Drama, Thriller"
        "Comedy"
        "Drama, Romance"
    
    This function returns a Series: ["Drama", "Thriller", "Comedy", "Drama", "Romance"]
    which we can then count, filter, or analyze.
    """
    return series.dropna().str.split(sep).explode().str.strip()

# Quick demonstration: how many unique genres exist?
all_genres = split_field(df['genres'])
print(f"Total genre tags across all movies: {len(all_genres):,}")
print(f"Unique genres: {all_genres.nunique()}")
print(f"\nAll unique genres: {sorted(all_genres.unique())}")

In [ ]:
# Create derived columns that will be useful throughout the analysis

# How many genres does each movie belong to?
df['genre_count'] = df['genres'].str.split(', ').apply(lambda x: len(x) if isinstance(x, list) else 0)

# How many countries were involved in production?
df['country_count'] = df['allCountries'].str.split(', ').apply(lambda x: len(x) if isinstance(x, list) else 0)

# Is this a co-production? (involves more than one country)
df['is_coproduction'] = df['country_count'] > 1

# How many keywords does each movie have?
df['keyword_count'] = df['keywords'].str.split(', ').apply(lambda x: len(x) if isinstance(x, list) else 0)

# Which decade was the movie released in? (useful for grouping)
df['decade'] = (df['releaseYear'] // 10) * 10

# Print a summary of our new columns
print("=== New Derived Columns ===\n")
print(f"  genre_count:     min={df['genre_count'].min()}, max={df['genre_count'].max()}, mean={df['genre_count'].mean():.1f}")
print(f"  country_count:   min={df['country_count'].min()}, max={df['country_count'].max()}, mean={df['country_count'].mean():.1f}")
print(f"  is_coproduction: {df['is_coproduction'].sum()} co-productions ({df['is_coproduction'].mean()*100:.1f}% of all movies)")
print(f"  keyword_count:   min={df['keyword_count'].min()}, max={df['keyword_count'].max()}, mean={df['keyword_count'].mean():.1f}")
print(f"  decade range:    {df['decade'].min()} — {df['decade'].max()}")

<a id="5-genres"></a>
## 5. Genre Analysis

Genres are central to how the film industry thinks about positioning. Publikum's PlotBounce tool uses 500 "expectation clusters" built from genre and plot data. While we cannot replicate that level of sophistication, we can map the **genre landscape** of our 2,000 European films.

**Questions we explore:**
- Which genres dominate European cinema?
- How often do genres co-occur (appear together on the same movie)?
- Are some genres associated with higher ratings than others?
- How many genres does a typical movie belong to?

In [ ]:
# Count how many movies each genre appears in
genre_counts = split_field(df['genres']).value_counts()

# Create a horizontal bar chart — easier to read genre names
fig, ax = plt.subplots(figsize=(12, 6))
genre_counts.plot(kind='barh', ax=ax, color=sns.color_palette('muted', len(genre_counts)))
ax.set_xlabel('Number of Movies')
ax.set_ylabel('Genre')
ax.set_title('Genre Distribution in European Films (n=2,000)')
ax.invert_yaxis()  # Put the most common genre at the top

# Add count labels at the end of each bar
for i, (val, name) in enumerate(zip(genre_counts.values, genre_counts.index)):
    ax.text(val + 10, i, str(val), va='center', fontsize=9)

plt.tight_layout()
save_figure(fig, 'genre_distribution')
plt.show()

### Genre co-occurrence: Which genres appear together?

A single movie can belong to multiple genres. Understanding which genres frequently **co-occur** reveals the "mental model" of genre combinations in European cinema. This is a form of **relational analysis** — we are looking at connections between categories, not just counting them individually.

In [ ]:
# Build a genre co-occurrence matrix
# For every movie, we look at all pairs of genres it belongs to and count them
genres_per_movie = df['genres'].dropna().str.split(', ')
unique_genres = sorted(split_field(df['genres']).unique())

# Initialize an empty matrix
cooccurrence = pd.DataFrame(0, index=unique_genres, columns=unique_genres)

# Count how often each pair of genres appears on the same movie
for genre_list in genres_per_movie:
    for g1, g2 in combinations(genre_list, 2):
        g1, g2 = g1.strip(), g2.strip()
        cooccurrence.loc[g1, g2] += 1
        cooccurrence.loc[g2, g1] += 1

# Visualize as a heatmap (show only the lower triangle to avoid redundancy)
fig, ax = plt.subplots(figsize=(14, 10))
mask = np.triu(np.ones_like(cooccurrence, dtype=bool), k=1)
sns.heatmap(cooccurrence, mask=mask, annot=True, fmt='d', cmap='Blues', ax=ax,
            linewidths=0.5, square=True)
ax.set_title('Genre Co-occurrence Matrix\nHow often do two genres appear on the same movie?')
plt.tight_layout()
save_figure(fig, 'genre_cooccurrence_heatmap')
plt.show()

In [ ]:
# Average IMDb rating by genre
# For each genre, we compute the mean rating of all movies containing that genre
genre_ratings = {}
for genre in unique_genres:
    mask = df['genres'].str.contains(genre, na=False)
    genre_ratings[genre] = {
        'mean_rating': df.loc[mask, 'imdbRating'].mean(),
        'median_rating': df.loc[mask, 'imdbRating'].median(),
        'count': mask.sum()
    }

genre_rating_df = pd.DataFrame(genre_ratings).T.sort_values('mean_rating', ascending=True)

fig, ax = plt.subplots(figsize=(10, 6))
# Color bars red if below median, green if above
median_rating = genre_rating_df['mean_rating'].median()
colors = ['#e74c3c' if r < median_rating else '#2ecc71' for r in genre_rating_df['mean_rating']]
ax.barh(genre_rating_df.index, genre_rating_df['mean_rating'], color=colors)
ax.axvline(df['imdbRating'].mean(), color='black', linestyle='--', alpha=0.5, 
           label=f'Overall mean: {df["imdbRating"].mean():.2f}')
ax.set_xlabel('Mean IMDb Rating')
ax.set_title('Average IMDb Rating by Genre')
ax.legend()
plt.tight_layout()
save_figure(fig, 'rating_by_genre')
plt.show()

In [ ]:
# How many genres does each movie belong to?
fig, ax = plt.subplots(figsize=(8, 5))
df['genre_count'].value_counts().sort_index().plot(kind='bar', ax=ax, color='steelblue')
ax.set_xlabel('Number of Genres per Movie')
ax.set_ylabel('Number of Movies')
ax.set_title('How Many Genres Does Each Movie Belong To?')
plt.tight_layout()
save_figure(fig, 'genres_per_movie')
plt.show()

### Interpretation: Genre landscape

**Observations:** *(Fill in after running the cells above)*
- Which genres dominate European cinema?
- Which genre combinations are most common (from the co-occurrence heatmap)?
- Do certain genres rate higher — and if so, why might that be?

> **Critical awareness:** Genre labels are not neutral. They are industry conventions that shape what gets produced, funded, and discovered. A movie labeled "Drama" is positioned differently from one labeled "Thriller" even if the content overlaps significantly. The genre taxonomy itself is a product of datafication. Also note: niche genres with few but dedicated IMDb voters may appear artificially high-rated.

<a id="6-countries"></a>
## 6. Country and Market Analysis

One of Publikum's core services is helping position films across different European markets. Understanding which countries produce what kinds of films — and how they collaborate — is essential for market strategy.

**Questions we explore:**
- Which countries are most represented in the dataset?
- Do different countries specialize in different genres?
- How common are international co-productions?
- Do co-productions perform differently than single-country films?

In [ ]:
# Count movies by main country
country_counts = df['mainCountry'].value_counts()

fig, ax = plt.subplots(figsize=(12, 8))
top_20 = country_counts.head(20)
top_20.plot(kind='barh', ax=ax, color='steelblue')
ax.set_xlabel('Number of Movies')
ax.set_ylabel('Main Country')
ax.set_title(f'Top 20 Countries by Number of Movies (total: {len(df):,})')
ax.invert_yaxis()

# Add count labels
for i, val in enumerate(top_20.values):
    ax.text(val + 5, i, str(val), va='center', fontsize=9)

plt.tight_layout()
save_figure(fig, 'movies_by_country')
plt.show()

In [ ]:
# Genre distribution by country — what does each country specialize in?
# We look at the top 10 countries and top 10 genres

top_countries = country_counts.head(10).index.tolist()
top_genres = split_field(df['genres']).value_counts().head(10).index.tolist()

# Build a country x genre matrix
country_genre = pd.DataFrame(0, index=top_countries, columns=top_genres)
for _, row in df[df['mainCountry'].isin(top_countries)].iterrows():
    if pd.notna(row['genres']):
        for genre in row['genres'].split(', '):
            genre = genre.strip()
            if genre in top_genres:
                country_genre.loc[row['mainCountry'], genre] += 1

# Normalize by country: what % of each country's movies fall into each genre?
country_genre_pct = country_genre.div(country_genre.sum(axis=1), axis=0) * 100

fig, ax = plt.subplots(figsize=(12, 8))
sns.heatmap(country_genre_pct, annot=True, fmt='.0f', cmap='YlOrRd', ax=ax)
ax.set_title("Genre Distribution by Country (%)\nWhat percentage of each country's films fall into each genre?")
ax.set_ylabel('Country')
ax.set_xlabel('Genre')
plt.tight_layout()
save_figure(fig, 'genre_by_country_heatmap')
plt.show()

In [ ]:
# Co-production analysis: how common are they, and do they perform differently?

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left panel: how many countries are involved per production?
df['country_count'].value_counts().sort_index().plot(kind='bar', ax=axes[0], color='steelblue')
axes[0].set_xlabel('Number of Countries Involved')
axes[0].set_ylabel('Number of Movies')
axes[0].set_title('How Many Countries Per Production?')

# Right panel: do co-productions rate differently?
single = df[~df['is_coproduction']]['imdbRating']
coprod = df[df['is_coproduction']]['imdbRating']
axes[1].boxplot([single.dropna(), coprod.dropna()], 
                labels=['Single Country', 'Co-production'])
axes[1].set_ylabel('IMDb Rating')
axes[1].set_title(f'Ratings: Single Country (n={len(single)}) vs Co-production (n={len(coprod)})')

plt.tight_layout()
save_figure(fig, 'coproduction_analysis')
plt.show()

# Print summary statistics
print(f"\nMean rating — Single country: {single.mean():.2f}")
print(f"Mean rating — Co-production:  {coprod.mean():.2f}")
print(f"Mean votes — Single country:  {df[~df['is_coproduction']]['numberOfVotes'].mean():,.0f}")
print(f"Mean votes — Co-production:   {df[df['is_coproduction']]['numberOfVotes'].mean():,.0f}")

In [ ]:
# Rating distribution by country (only countries with enough movies for a meaningful comparison)
min_movies = 20
country_with_enough = country_counts[country_counts >= min_movies].index

fig, ax = plt.subplots(figsize=(14, 6))
country_data = df[df['mainCountry'].isin(country_with_enough)]
# Order by median rating (highest first)
order = country_data.groupby('mainCountry')['imdbRating'].median().sort_values(ascending=False).index
sns.boxplot(data=country_data, x='mainCountry', y='imdbRating', order=order, ax=ax, palette='muted')
ax.set_xticklabels(ax.get_xticklabels(), rotation=45, ha='right')
ax.set_title(f'IMDb Rating Distribution by Country (min {min_movies} movies)')
ax.set_xlabel('Main Country')
ax.set_ylabel('IMDb Rating')
plt.tight_layout()
save_figure(fig, 'rating_by_country')
plt.show()

### Interpretation: Country patterns

**Observations:** *(Fill in after running the cells above)*
- Which countries dominate the dataset?
- Are there genre specializations (e.g., Nordic crime, French comedy, Italian drama)?
- How do co-productions compare to single-country films in terms of ratings and visibility?

> **Critical awareness:** The `mainCountry` field is itself an analytical choice made during data structuring. For a French-German-Belgian co-production, who decides which country is "main"? This classification shapes what patterns we can see. Also note: the dataset contains only 2,000 titles and may significantly over- or under-represent certain national cinemas.

<a id="7-ratings"></a>
## 7. Ratings and Popularity Analysis

IMDb ratings and vote counts are the primary quantitative measures in our dataset. But they are **proxies**, not direct measures of quality or commercial success. Understanding their distributions — and their relationship to each other — helps us use them wisely.

**Questions we explore:**
- What does the rating distribution look like?
- What is the relationship between ratings and number of votes?
- Can we identify different "quadrants" of films (popular vs. niche, high-rated vs. low-rated)?

In [ ]:
# Distribution of ratings and vote counts

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left: histogram of IMDb ratings
axes[0].hist(df['imdbRating'].dropna(), bins=30, color='steelblue', edgecolor='white', alpha=0.8)
axes[0].axvline(df['imdbRating'].mean(), color='red', linestyle='--', 
                label=f'Mean: {df["imdbRating"].mean():.2f}')
axes[0].axvline(df['imdbRating'].median(), color='orange', linestyle='--', 
                label=f'Median: {df["imdbRating"].median():.2f}')
axes[0].set_xlabel('IMDb Rating')
axes[0].set_ylabel('Number of Movies')
axes[0].set_title('Distribution of IMDb Ratings')
axes[0].legend()

# Right: histogram of vote counts (log scale because votes are very skewed)
axes[1].hist(df['numberOfVotes'].dropna(), bins=50, color='coral', edgecolor='white', alpha=0.8)
axes[1].set_xlabel('Number of Votes')
axes[1].set_ylabel('Number of Movies')
axes[1].set_title('Distribution of Vote Counts (log scale)')
axes[1].set_xscale('log')

plt.tight_layout()
save_figure(fig, 'rating_vote_distributions')
plt.show()

In [ ]:
# The Four Quadrants: Rating vs. Popularity scatter plot
# This is one of the most informative EDA visualizations — it reveals the
# relationship between "quality" (rating) and "visibility" (votes)

fig, ax = plt.subplots(figsize=(10, 8))

scatter = ax.scatter(df['numberOfVotes'], df['imdbRating'], 
                     alpha=0.3, s=20, c=df['genre_count'], cmap='viridis')
ax.set_xscale('log')
ax.set_xlabel('Number of Votes (log scale)')
ax.set_ylabel('IMDb Rating')
ax.set_title('Rating vs. Popularity: The Four Quadrants of European Film')

# Draw quadrant dividers at the median values
median_votes = df['numberOfVotes'].median()
median_rating = df['imdbRating'].median()
ax.axhline(median_rating, color='gray', linestyle='--', alpha=0.5)
ax.axvline(median_votes, color='gray', linestyle='--', alpha=0.5)

# Label the four quadrants
ax.text(0.02, 0.98, 'Niche\nCritical Acclaim', transform=ax.transAxes, fontsize=10, 
        va='top', ha='left', style='italic', color='gray')
ax.text(0.98, 0.98, 'Popular\nCritical Acclaim', transform=ax.transAxes, fontsize=10, 
        va='top', ha='right', style='italic', color='gray')
ax.text(0.02, 0.02, 'Niche\nLow-Rated', transform=ax.transAxes, fontsize=10, 
        va='bottom', ha='left', style='italic', color='gray')
ax.text(0.98, 0.02, 'Popular\nPolarizing', transform=ax.transAxes, fontsize=10, 
        va='bottom', ha='right', style='italic', color='gray')

plt.colorbar(scatter, label='Number of Genres')
plt.tight_layout()
save_figure(fig, 'rating_vs_popularity_quadrants')
plt.show()

In [ ]:
# Top-rated and most-voted films — let's see who's at the extremes

print("=== Top 10 Highest-Rated Films (min 1,000 votes) ===\n")
top_rated = df[df['numberOfVotes'] >= 1000].nlargest(10, 'imdbRating')
print(top_rated[['originalTitle', 'releaseYear', 'imdbRating', 'numberOfVotes', 
                  'mainCountry', 'genres']].to_string(index=False))

print("\n\n=== Top 10 Most-Voted Films ===\n")
most_voted = df.nlargest(10, 'numberOfVotes')
print(most_voted[['originalTitle', 'releaseYear', 'imdbRating', 'numberOfVotes', 
                   'mainCountry', 'genres']].to_string(index=False))

### Interpretation: Ratings and popularity

**Observations:** *(Fill in after running the cells above)*
- Is the rating distribution roughly normal? Skewed?
- What is the relationship between votes and ratings?
- What kinds of films appear in each quadrant of the scatter plot?
- Are the top-rated and most-voted films the same, or different?

> **Critical awareness:** IMDb's user base skews male, English-speaking, and younger (Sapienza & Lehmann, 2021). A highly-rated film on IMDb is not necessarily well-received by general European audiences. The number of votes reflects IMDb engagement, not box office success or streaming numbers. Films in non-English languages may be systematically under-voted.

<a id="8-temporal"></a>
## 8. Temporal Trends

How has European cinema changed over time? Temporal analysis can reveal shifts in genre popularity, production volume, and quality patterns across decades.

In [ ]:
# Production volume and average rating over time

fig, axes = plt.subplots(2, 1, figsize=(14, 10))

# Top panel: number of movies per year
year_counts = df['releaseYear'].value_counts().sort_index()
axes[0].bar(year_counts.index, year_counts.values, color='steelblue', alpha=0.7)
axes[0].set_xlabel('Release Year')
axes[0].set_ylabel('Number of Movies')
axes[0].set_title('Production Volume Over Time')

# Bottom panel: average rating by decade
decade_stats = df.groupby('decade').agg(
    mean_rating=('imdbRating', 'mean'),
    median_rating=('imdbRating', 'median'),
    count=('imdbRating', 'count')
).reset_index()

axes[1].plot(decade_stats['decade'], decade_stats['mean_rating'], 'o-', 
             color='steelblue', label='Mean rating')
axes[1].plot(decade_stats['decade'], decade_stats['median_rating'], 's--', 
             color='coral', label='Median rating')
axes[1].set_xlabel('Decade')
axes[1].set_ylabel('IMDb Rating')
axes[1].set_title('Average Rating by Decade')
axes[1].legend()

# Add sample size annotations so we know how reliable each decade is
for _, row in decade_stats.iterrows():
    axes[1].annotate(f'n={int(row["count"])}', (row['decade'], row['mean_rating']),
                    textcoords="offset points", xytext=(0, 10), ha='center', fontsize=8)

plt.tight_layout()
save_figure(fig, 'temporal_trends')
plt.show()

In [ ]:
# Genre trends over decades — how have genre shares shifted?
# We track the top 6 genres from the 1960s onward (earlier decades have too few movies)

top_6_genres = split_field(df['genres']).value_counts().head(6).index.tolist()
recent_decades = df[df['decade'] >= 1960]

genre_decade = pd.DataFrame()
for genre in top_6_genres:
    mask = recent_decades['genres'].str.contains(genre, na=False)
    counts = recent_decades[mask].groupby('decade').size()
    totals = recent_decades.groupby('decade').size()
    genre_decade[genre] = (counts / totals * 100).round(1)

fig, ax = plt.subplots(figsize=(12, 6))
genre_decade.plot(ax=ax, marker='o')
ax.set_xlabel('Decade')
ax.set_ylabel('% of Movies in Decade')
ax.set_title('Genre Trends Over Time (as % of movies per decade)')
ax.legend(title='Genre', bbox_to_anchor=(1.05, 1), loc='upper left')
plt.tight_layout()
save_figure(fig, 'genre_trends_over_time')
plt.show()

<a id="9-runtime"></a>
## 9. Runtime Analysis

Runtime conventions vary by genre and have shifted over time. Understanding these patterns can inform project positioning — a 150-minute drama signals something different from a 90-minute thriller.

In [ ]:
# Runtime analysis: overall distribution and by genre

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Left: overall runtime distribution
axes[0].hist(df['runtimeMinutes'].dropna(), bins=40, color='steelblue', edgecolor='white', alpha=0.8)
axes[0].axvline(df['runtimeMinutes'].mean(), color='red', linestyle='--', 
                label=f'Mean: {df["runtimeMinutes"].mean():.0f} min')
axes[0].axvline(df['runtimeMinutes'].median(), color='orange', linestyle='--', 
                label=f'Median: {df["runtimeMinutes"].median():.0f} min')
axes[0].set_xlabel('Runtime (minutes)')
axes[0].set_ylabel('Number of Movies')
axes[0].set_title('Runtime Distribution')
axes[0].legend()

# Right: runtime by genre (top 8 genres, as boxplots)
top_8_genres = split_field(df['genres']).value_counts().head(8).index.tolist()
genre_runtime_data = []
for genre in top_8_genres:
    mask = df['genres'].str.contains(genre, na=False)
    for val in df.loc[mask, 'runtimeMinutes'].dropna():
        genre_runtime_data.append({'genre': genre, 'runtime': val})

genre_rt_df = pd.DataFrame(genre_runtime_data)
order = genre_rt_df.groupby('genre')['runtime'].median().sort_values(ascending=False).index
sns.boxplot(data=genre_rt_df, x='genre', y='runtime', order=order, ax=axes[1], palette='muted')
axes[1].set_xticklabels(axes[1].get_xticklabels(), rotation=45, ha='right')
axes[1].set_xlabel('Genre')
axes[1].set_ylabel('Runtime (minutes)')
axes[1].set_title('Runtime by Genre')

plt.tight_layout()
save_figure(fig, 'runtime_analysis')
plt.show()

<a id="10-keywords"></a>
## 10. Keyword and Theme Analysis

The `keywords` column contains user-contributed tags describing themes, settings, and motifs. While not standardized (an important caveat), they offer a window into the **thematic landscape** of European cinema.

This analysis connects to Publikum's **PlotBounce** tool, which uses 500 "expectation clusters" built from 163,000 films. We cannot replicate that sophistication, but we can explore keyword patterns in our 2,000-film subset.

> **Datafication moment:** Keywords are user-contributed and not standardized. "world war ii," "world war two," and "ww2" might all refer to the same concept. Our analysis inherits these inconsistencies.

In [ ]:
# Top keywords across all movies
all_keywords = split_field(df['keywords'])
keyword_counts = all_keywords.value_counts()

print(f"Total keyword tags: {len(all_keywords):,}")
print(f"Unique keywords: {keyword_counts.shape[0]:,}")

fig, ax = plt.subplots(figsize=(10, 8))
keyword_counts.head(30).plot(kind='barh', ax=ax, color='steelblue')
ax.set_xlabel('Frequency')
ax.set_title('Top 30 Keywords Across European Films')
ax.invert_yaxis()
plt.tight_layout()
save_figure(fig, 'top_keywords')
plt.show()

In [ ]:
# What keywords characterize each genre?
# For selected genres, we find their most common keywords

print("=== Most Common Keywords by Genre ===\n")
for genre in ['Drama', 'Comedy', 'Thriller', 'Horror', 'Romance']:
    mask = df['genres'].str.contains(genre, na=False)
    genre_keywords = split_field(df.loc[mask, 'keywords']).value_counts().head(10)
    print(f"\n{genre} (top 10 keywords):")
    for kw, count in genre_keywords.items():
        print(f"  {kw}: {count}")

In [ ]:
# Keyword co-occurrence heatmap for the top 30 keywords
# This shows which themes tend to appear together, leading into our network analysis

top_kw = keyword_counts.head(30).index.tolist()

kw_cooccurrence = pd.DataFrame(0, index=top_kw, columns=top_kw)
for _, row in df.dropna(subset=['keywords']).iterrows():
    kws = [k.strip() for k in row['keywords'].split(', ') if k.strip() in top_kw]
    for k1, k2 in combinations(kws, 2):
        kw_cooccurrence.loc[k1, k2] += 1
        kw_cooccurrence.loc[k2, k1] += 1

fig, ax = plt.subplots(figsize=(14, 12))
mask = np.triu(np.ones_like(kw_cooccurrence, dtype=bool), k=1)
sns.heatmap(kw_cooccurrence, mask=mask, cmap='YlOrRd', ax=ax, linewidths=0.5)
ax.set_title('Keyword Co-occurrence (Top 30 Keywords)\nWhich themes tend to appear together?')
plt.tight_layout()
save_figure(fig, 'keyword_cooccurrence')
plt.show()

<a id="11-network"></a>
## 11. Co-production Network Analysis

This section applies **Visual Network Analysis (VNA)** as taught in the course (Venturini et al., 2021). Networks reveal **relational structures** that are invisible in tables and charts.

We construct a **country co-production network** where:
- **Nodes** = countries
- **Edges** = shared movie productions (two countries are connected if they co-produced at least one film)
- **Edge weight** = number of co-productions between two countries

This directly addresses a key decision question for Publikum: **Which markets or countries offer the best fit for collaboration?**

> **Relational ambiguity (Venturini et al., 2021):** The same data can produce different networks depending on how we define nodes and edges. Our network shows co-production *frequency*, not co-production *success* or cultural affinity. A different definition would reveal different patterns.

In [ ]:
# Build the co-production network
# Each country is a node; an edge connects two countries that co-produced a film

G_country = nx.Graph()

for _, row in df.dropna(subset=['allCountries']).iterrows():
    countries = [c.strip() for c in row['allCountries'].split(', ')]
    if len(countries) > 1:  # Only co-productions create edges
        for c1, c2 in combinations(countries, 2):
            if G_country.has_edge(c1, c2):
                G_country[c1][c2]['weight'] += 1
            else:
                G_country.add_edge(c1, c2, weight=1)

# Add node attribute: how many movies is this country the "main" country for?
for node in G_country.nodes():
    G_country.nodes[node]['movie_count'] = (df['mainCountry'] == node).sum()

print(f"Network: {G_country.number_of_nodes()} countries, {G_country.number_of_edges()} co-production links")
print(f"\nMost connected countries (by number of partners):")
for node, degree in sorted(G_country.degree(), key=lambda x: x[1], reverse=True)[:10]:
    total_weight = sum(G_country[node][n]['weight'] for n in G_country.neighbors(node))
    print(f"  {node}: {degree} partners, {total_weight} total co-productions")

In [ ]:
# Visualize the co-production network using a force-directed layout
# This is similar to ForceAtlas2 from the Gephi exercises in the course

fig, ax = plt.subplots(figsize=(16, 12))

# Spring layout: nodes that share more co-productions are pulled closer together
pos = nx.spring_layout(G_country, k=2, iterations=50, seed=42, weight='weight')

# Node size proportional to how many partners a country has (degree)
node_sizes = [G_country.degree(n) * 50 + 100 for n in G_country.nodes()]

# Edge width proportional to number of co-productions (weight)
edge_weights = [G_country[u][v]['weight'] for u, v in G_country.edges()]
max_weight = max(edge_weights) if edge_weights else 1
edge_widths = [w / max_weight * 5 + 0.5 for w in edge_weights]

# Draw the network
nx.draw_networkx_edges(G_country, pos, alpha=0.3, width=edge_widths, edge_color='gray', ax=ax)
nx.draw_networkx_nodes(G_country, pos, node_size=node_sizes, node_color='steelblue', alpha=0.7, ax=ax)
nx.draw_networkx_labels(G_country, pos, font_size=8, font_weight='bold', ax=ax)

ax.set_title('Country Co-production Network\n'
             'Nodes = countries, Edges = shared film productions\n'
             '(Layout: force-directed / spring — frequent collaborators are closer)', 
             fontsize=14)
ax.axis('off')
plt.tight_layout()
save_figure(fig, 'coproduction_network')
plt.show()

In [ ]:
# Network centrality metrics — who are the most important countries in the network?

print("=== Betweenness Centrality ===")
print("(Countries that bridge different parts of the network — gatekeepers)\n")
betweenness = nx.betweenness_centrality(G_country, weight='weight')
for node, score in sorted(betweenness.items(), key=lambda x: x[1], reverse=True)[:10]:
    print(f"  {node}: {score:.4f}")

print("\n=== Weighted Degree Centrality ===")
print("(Total co-production volume — most active collaborators)\n")
w_degree = {n: sum(G_country[n][nbr]['weight'] for nbr in G_country.neighbors(n)) 
            for n in G_country.nodes()}
for node, score in sorted(w_degree.items(), key=lambda x: x[1], reverse=True)[:10]:
    print(f"  {node}: {score} co-productions")

### Interpretation: Co-production landscape

**Observations:** *(Fill in after running the cells above)*
- Which countries are most central in the co-production network?
- Are there visible clusters (e.g., Nordic, Western European, Eastern European)?
- Which countries serve as "bridges" between clusters?
- How does this relate to Publikum's Scandinavian focus?

**Course connection — VNA concepts applied:**
- **Degree** = number of co-production partners (breadth of collaboration)
- **Betweenness centrality** = countries that bridge otherwise disconnected groups (gatekeepers)
- **Force-directed layout** = spatially positions countries so that frequent collaborators are closer together (similar to ForceAtlas2 from the course exercises)

<a id="12-genre-network"></a>
## 12. Genre Co-occurrence Network

A second application of VNA: treating **genres as nodes**, connected by how often they appear together on the same film. This reveals the "genre space" of European cinema — which genres are close neighbors and which are worlds apart.

In [ ]:
# Build and visualize the genre co-occurrence network
G_genre = nx.Graph()

for _, row in df.dropna(subset=['genres']).iterrows():
    genres = [g.strip() for g in row['genres'].split(', ')]
    for g1, g2 in combinations(genres, 2):
        if G_genre.has_edge(g1, g2):
            G_genre[g1][g2]['weight'] += 1
        else:
            G_genre.add_edge(g1, g2, weight=1)

# Node attribute: how many movies contain this genre
for node in G_genre.nodes():
    G_genre.nodes[node]['count'] = (df['genres'].str.contains(node, na=False)).sum()

fig, ax = plt.subplots(figsize=(12, 10))
pos = nx.spring_layout(G_genre, k=3, iterations=100, seed=42, weight='weight')

# Node size proportional to genre frequency
node_sizes = [G_genre.nodes[n]['count'] * 3 for n in G_genre.nodes()]

# Edge width proportional to co-occurrence
edge_weights = [G_genre[u][v]['weight'] for u, v in G_genre.edges()]
max_w = max(edge_weights)
edge_widths = [w / max_w * 8 + 0.5 for w in edge_weights]

nx.draw_networkx_edges(G_genre, pos, alpha=0.3, width=edge_widths, edge_color='gray', ax=ax)
nx.draw_networkx_nodes(G_genre, pos, node_size=node_sizes, node_color='coral', alpha=0.7, ax=ax)
nx.draw_networkx_labels(G_genre, pos, font_size=10, font_weight='bold', ax=ax)

# Label the strongest connections
top_edges = sorted(G_genre.edges(data=True), key=lambda x: x[2]['weight'], reverse=True)[:5]
edge_labels = {(u, v): d['weight'] for u, v, d in top_edges}
nx.draw_networkx_edge_labels(G_genre, pos, edge_labels, font_size=8, ax=ax)

ax.set_title('Genre Co-occurrence Network\n'
             'Node size = genre frequency, Edge thickness = co-occurrence count\n'
             'Numbers show the 5 strongest connections')
ax.axis('off')
plt.tight_layout()
save_figure(fig, 'genre_network')
plt.show()

<a id="13-comparable"></a>
## 13. Comparable Title Analysis

This section demonstrates a simplified version of what Publikum's **PlotBounce** tool does: given a hypothetical new film project, find **comparable existing titles** based on genre, keywords, and market.

This is the most directly **actionable** part of our analysis — it connects analytical findings to the decision question: *How should a film project be positioned?*

In [ ]:
# Example scenario: Finding comparable titles for a new Scandinavian crime-drama
# This uses the find_comparable_titles function from src/analysis.py

print("=== Comparable Title Search ===")
print("Scenario: A new Scandinavian crime-drama targeting international audiences\n")

# Find top-rated Crime/Drama films with significant audience engagement
target_genres = ['Crime', 'Drama']
comparable = find_comparable_titles(df, target_genres, min_votes=1000, top_n=15)

print(f"Top 15 Crime/Drama films with 1,000+ votes:\n")
print(comparable[['originalTitle', 'releaseYear', 'imdbRating', 'numberOfVotes', 
                   'mainCountry', 'genres']].to_string(index=False))

# Also look specifically at Nordic films
target_countries = ['DK', 'SE', 'NO', 'FI', 'IS']
nordic_mask = (df['genres'].str.contains('Crime', na=False) & 
               df['mainCountry'].isin(target_countries))
nordic_crime = df[nordic_mask].nlargest(10, 'numberOfVotes')

print(f"\n\n=== Nordic Crime Films (top 10 by visibility) ===\n")
print(nordic_crime[['originalTitle', 'releaseYear', 'imdbRating', 'numberOfVotes', 
                      'mainCountry', 'genres']].to_string(index=False))

In [ ]:
# Visualize where comparable titles sit in the overall landscape

fig, ax = plt.subplots(figsize=(10, 8))

# All movies as light background dots
ax.scatter(df['numberOfVotes'], df['imdbRating'], alpha=0.1, s=10, color='gray', label='All movies')

# Highlight comparable titles in red
ax.scatter(comparable['numberOfVotes'], comparable['imdbRating'], 
           s=80, color='red', zorder=5, edgecolors='black', label='Comparable titles')

# Label each comparable title
for _, row in comparable.iterrows():
    ax.annotate(row['originalTitle'][:25], (row['numberOfVotes'], row['imdbRating']),
               fontsize=7, alpha=0.8, xytext=(5, 5), textcoords='offset points')

ax.set_xscale('log')
ax.set_xlabel('Number of Votes (log scale)')
ax.set_ylabel('IMDb Rating')
ax.set_title('Comparable Crime/Drama Films Highlighted in Context')
ax.legend()
plt.tight_layout()
save_figure(fig, 'comparable_titles_context')
plt.show()

<a id="14-summary"></a>
## 14. Summary of Findings and Limitations

### Key Findings

*(Fill in after running all sections. Structure around the decision questions:)*

1. **Genre landscape:** *(Summary of dominant genres, co-occurrences, rating patterns)*
2. **Country patterns:** *(Which countries dominate, genre specializations, co-production dynamics)*
3. **Rating and popularity:** *(What characterizes high-rated vs. popular films, the four quadrants)*
4. **Temporal trends:** *(How has the landscape shifted over time)*
5. **Keyword themes:** *(What thematic patterns emerge)*
6. **Network structures:** *(What co-production clusters exist, who bridges them)*
7. **Comparable titles:** *(Demonstration of the approach for a Nordic crime-drama)*

---

### Limitations and Critical Reflections

Following the course frameworks, we must be transparent about what this analysis does **not** capture:

| Limitation | Impact | Course framework |
|-----------|--------|-----------------|
| IMDb ratings reflect a biased user base (male, English-speaking, younger) | May misrepresent reception in non-English markets | Datafication — Production (Flyverbom & Madsen) |
| Number of votes is not box office success | High IMDb visibility ≠ commercial viability | Critical data awareness (Mejias & Couldry) |
| We only have the References pillar | Missing Story (audience reactions) and Zeitgeist (cultural relevance) | Publikum methodology |
| Genre labels are industry conventions, not natural categories | Our genre analysis is shaped by how IMDb categorizes | Datafication — Structuring |
| Keywords are user-contributed and inconsistent | Keyword analysis may miss or conflate themes | Data quality |
| 2,000 titles is a sample, not the full landscape | Country and genre distributions may not represent all European cinema | Statistical representativeness |
| The dataset is a snapshot in time | Ratings, votes, and classifications change | Datafication — Distribution |

---

### What Publikum's other pillars would add

Our analysis provides a **References-level** view. To move from analytics to action, Publikum would layer on:

- **Story:** How do ~8,000 test audience members emotionally react to specific narrative elements? (Mobile research, video-based responses)
- **Zeitgeist:** What cultural conversations is a film tapping into? (Social media monitoring — e.g., how is "narcissism" discussed differently in Scandinavia vs. Southern Europe)

As Publikum frames it: these are *"not conclusions, but structured reflections."* Our data analysis is one input into a much richer decision-making process.

> *"Far better an approximate answer to the right question, which is often vague, than an exact answer to the wrong question, which can always be made precise."* — John Tukey, 1977